# Notebook 32 — Frozen-body gate-depth × gate-LR study

Runs the complete paired 2×2 factorial for the strongest compact-MoE direction: the exact Notebook 31 encoder and Stage-B linear experts remain frozen throughout Stage C, while only gate depth (`64 → 4` versus dropout-free `64 → 32 → 4`) and gate learning rate (`1e-3` versus `3e-4`) vary. All four cells reuse the same signed Stage-A/B artifacts, paired seeds 0–2, splits, load-balancing coefficient, epoch budget, validation selection, and locked-test reporting. The already completed deeper-gate/`1e-3` cell is reused under its immutable prefix; only the three missing cells require new Stage-C runs.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
BASELINE_PREFIX = 'nfv3_4way_moe_soft_comparable_repro_v1'
STUDIES = [
    ('linear_lr1e3', 'config/compact_soft_moe_frozen_stagec_linear_gate_lr1e3_3seed.yaml', 'nfv3_4way_moe_soft_linear_gate_frozen_stagec_lr1e3_v1'),
    ('linear_lr3e4', 'config/compact_soft_moe_frozen_stagec_linear_gate_lr3e4_3seed.yaml', 'nfv3_4way_moe_soft_linear_gate_frozen_stagec_lr3e4_v1'),
    ('deep32_lr1e3', 'config/compact_soft_moe_deeper_gate_frozen_stagec_3seed.yaml', 'nfv3_4way_moe_soft_deeper_gate32_frozen_stagec_v1'),
    ('deep32_lr3e4', 'config/compact_soft_moe_frozen_stagec_deeper_gate_lr3e4_3seed.yaml', 'nfv3_4way_moe_soft_deeper_gate32_frozen_stagec_lr3e4_v1'),
]
EXECUTE = False  # Run the dry-run first; then change to True.
MAX_NEW_SEEDS_PER_STUDY = 3  # Set 1 to advance each incomplete cell one seed per session.
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token: raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}'}
repo = Path('/content') / GITHUB_REPO; url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir(): subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else: subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None; os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR; os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'; os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
print('Repository commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

## Contract preflight, dry-run, and resumable execution

Every cell must schedule Stage C only, load the exact same baseline Stage-A/B checkpoints, freeze the shared encoder and complete expert bank, and leave only either 260 linear-gate parameters or 2,212 nonlinear-gate parameters trainable.

In [ ]:
import torch, yaml
from training.logging_utils import run_streaming_logged
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime before executing the study.')
expected = {
    'linear_lr1e3': ([], 0.001, 20380),
    'linear_lr3e4': ([], 0.0003, 20380),
    'deep32_lr1e3': ([32], 0.001, 22332),
    'deep32_lr3e4': ([32], 0.0003, 22332),
}
for condition, study_config, _prefix in STUDIES:
    study = yaml.safe_load(Path(study_config).read_text())
    gate_dims, stage_c_lr, parameters = expected[condition]
    assert study['seeds'] == [0, 1, 2]
    assert study['reuse_stage_a_from_prefix'] == study['reuse_stage_b_from_prefix'] == BASELINE_PREFIX
    assert study['backbone']['model']['encoder']['hidden_dims'] == [128]
    assert study['backbone']['model']['expert']['hidden_dims'] == []
    assert study['backbone']['model']['gate']['hidden_dims'] == gate_dims
    assert study['backbone']['training']['stage_c_unfreeze'] == 'none'
    assert study['backbone']['training']['stage_c']['freeze_experts'] is True
    assert study['backbone']['training']['stage_c']['optimizer']['lr'] == stage_c_lr
    assert study['expected_total_parameters'] == parameters
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_global_residual_experts.py', 'tests/test_compact_soft_moe_run.py', 'tests/test_notebook31_ablation_methods.py', 'tests/test_dataset_blind_inference.py', 'tests/test_latent_space.py', 'tests/test_out_of_core.py'], check=True)
for condition, study_config, prefix in STUDIES:
    study = yaml.safe_load(Path(study_config).read_text())
    gate = study['backbone']['model']['gate']['hidden_dims']; lr = study['backbone']['training']['stage_c']['optimizer']['lr']
    summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary'; summary_dir.mkdir(parents=True, exist_ok=True)
    print(f'=== {condition}: gate={gate or "linear"}, Stage-C LR={lr:g}, parameters={study["expected_total_parameters"]:,} ===')
    command = [sys.executable, '-u', '-m', 'training.compact_soft_moe_run', '--study-config', study_config, '--prefix', prefix]
    run_streaming_logged(command, str(summary_dir / 'Notebook_Dry_Run.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'32-{condition}-dry-run')
    if EXECUTE:
        run_streaming_logged([*command, '--execute', '--max-new-seeds', str(MAX_NEW_SEEDS_PER_STUDY)], str(summary_dir / 'Notebook_Training_Stream.log'), env=os.environ.copy(), heartbeat_seconds=30, label=f'32-{condition}-training')
if not EXECUTE: print('NO TRAINING WAS STARTED. Review all four contracts, set EXECUTE=True, and rerun this cell.')

## Paired factorial analysis

The winning cell is selected exclusively by mean validation macro F1. Locked-test metrics remain descriptive and are displayed only after every three-seed manifest is complete.

In [ ]:
import numpy as np, pandas as pd
from IPython.display import Image, display
summaries = {condition: Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_summary' for condition, _, prefix in STUDIES}
complete = {name: (path / 'Final_Study_Manifest.json').is_file() for name, path in summaries.items()}; print('Completion:', complete)
winner_condition = None
if all(complete.values()):
    validation = []; test = []; classes = []; datasets = []; resources = []
    files = [('Three_Seed_Guardrails.csv', validation), ('Three_Seed_Overall.csv', test), ('Three_Seed_Per_Class.csv', classes), ('Three_Seed_Per_Dataset.csv', datasets), ('Three_Seed_Resource_Accounting.csv', resources)]
    for name, path in summaries.items():
        for filename, target in files:
            frame = pd.read_csv(path / filename); frame.insert(0, 'condition', name); target.append(frame)
    validation = pd.concat(validation, ignore_index=True); test = pd.concat(test, ignore_index=True)
    print('=== Validation guardrails ==='); display(validation[['condition', 'seed', 'validation_macro_f1', 'worst_dataset_macro_f1', 'rare_recall_mean', 'weakest_class', 'weakest_class_recall']])
    pivot = validation.pivot(index='seed', columns='condition', values='validation_macro_f1')
    pivot['depth_effect_lr1e3'] = pivot['deep32_lr1e3'] - pivot['linear_lr1e3']
    pivot['depth_effect_lr3e4'] = pivot['deep32_lr3e4'] - pivot['linear_lr3e4']
    pivot['lower_lr_effect_linear'] = pivot['linear_lr3e4'] - pivot['linear_lr1e3']
    pivot['lower_lr_effect_deep32'] = pivot['deep32_lr3e4'] - pivot['deep32_lr1e3']
    print('=== Paired validation effects ==='); display(pivot)
    means = validation.groupby('condition')['validation_macro_f1'].mean()
    depth_effect = ((means['deep32_lr1e3'] + means['deep32_lr3e4']) - (means['linear_lr1e3'] + means['linear_lr3e4'])) / 2
    lower_lr_effect = ((means['linear_lr3e4'] + means['deep32_lr3e4']) - (means['linear_lr1e3'] + means['deep32_lr1e3'])) / 2
    interaction = means['deep32_lr3e4'] - means['deep32_lr1e3'] - means['linear_lr3e4'] + means['linear_lr1e3']
    winner_condition = means.idxmax()
    print('Validation-selected winner:', winner_condition, 'mean macro F1=', means[winner_condition])
    print('Mean deeper-gate effect:', depth_effect); print('Mean lower-LR effect:', lower_lr_effect); print('Depth × LR interaction:', interaction)
    metrics = [c for c in ['accuracy', 'macro_f1', 'macro_precision', 'macro_recall', 'weighted_f1', 'roc_auc_ovr_macro', 'pr_auc_ovr_macro'] if c in test]
    print('=== Locked-test descriptive mean and SD ==='); display(test.groupby('condition')[metrics].agg(['mean', 'std']))
    print('=== Per-dataset macro F1 ==='); display(pd.concat(datasets, ignore_index=True).pivot_table(index='origin', columns='condition', values='macro_f1', aggfunc='mean'))
    print('=== Per-class F1 ==='); display(pd.concat(classes, ignore_index=True).pivot_table(index='class', columns='condition', values='f1', aggfunc='mean'))
    resource_frame = pd.concat(resources, ignore_index=True); display(resource_frame)
    expected_trainable = {'linear_lr1e3': 260, 'linear_lr3e4': 260, 'deep32_lr1e3': 2212, 'deep32_lr3e4': 2212}
    for condition, count in expected_trainable.items(): assert set(resource_frame.query('condition == @condition')['trainable_parameters'].astype(int)) == {count}
    baseline_summary = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{BASELINE_PREFIX}_summary' / 'Three_Seed_Overall.csv'
    if baseline_summary.is_file():
        baseline = pd.read_csv(baseline_summary); print('=== Original all-trainable Notebook 31 reference ==='); display(baseline[metrics + ['seed']])
else: print('Factorial tables appear after all four three-seed manifests are complete.')

## Freeze-integrity and routing diagnostics

All four cells must preserve the reused Stage-B encoder/expert functions exactly. Any nonzero owned-expert metric delta is treated as a contract failure. Routing diagnostics are then shown for the validation-selected cell.

In [ ]:
if all(complete.values()):
    preservation = []
    for condition, _study_config, prefix in STUDIES:
        for seed in (0, 1, 2):
            stage_b = pd.read_csv(Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / f'{BASELINE_PREFIX}_seed{seed}' / 'Validation_Expert_Owned_StageB.csv')
            stage_c = pd.read_csv(Path(DRIVE_OUTPUT_DIR) / 'results' / f'{prefix}_seed{seed}' / 'Validation_Expert_Owned_StageC.csv')
            merged = stage_b.merge(stage_c, on='dataset', suffixes=('_stage_b', '_stage_c'))
            delta_columns = []
            for metric in ('macro_f1', 'macro_precision', 'macro_recall', 'accuracy'):
                column = f'delta_{metric}'; merged[column] = merged[f'{metric}_stage_c'] - merged[f'{metric}_stage_b']; delta_columns.append(column)
            assert np.allclose(merged[delta_columns], 0.0, atol=1e-12), f'Frozen-body drift in {condition}, seed {seed}'
            merged.insert(0, 'seed', seed); merged.insert(0, 'condition', condition); preservation.append(merged)
    print('=== Exact Stage-B competence preservation ==='); display(pd.concat(preservation, ignore_index=True))
    winner_prefix = next(prefix for condition, _, prefix in STUDIES if condition == winner_condition)
    for seed in (0, 1, 2):
        result_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{winner_prefix}_seed{seed}'
        print(f'=== {winner_condition}, seed {seed}: gate by source dataset ==='); display(pd.read_csv(result_dir / 'Gate_By_Dataset.csv'))
        print(f'=== {winner_condition}, seed {seed}: expert utilization ==='); display(pd.read_csv(result_dir / 'Expert_Utilization.csv'))

## Stage-C stability and latent diagnostics

In [ ]:
if all(complete.values()):
    for name, path in summaries.items():
        print('===', name, 'Stage-C history ===')
        history = pd.read_csv(path / 'Training_History_3Seed.csv').query("stage == 'C'")
        display(history[['seed', 'epoch', 'learning_rate', 'train_total_loss', 'train_ce_loss', 'train_balance_penalty', 'val_macro_f1', 'best_val_macro_f1', 'patience_left']])
        display(pd.read_csv(path / 'Latent_Snapshot_Summary_3Seed.csv'))
        for figure in sorted((path / 'training_figures').glob('*.png')): display(Image(filename=str(figure)))